# 📘 Week 2 · Day 10-11: ML 기초 모델 & Scikit-learn 워크플로우

## 🎯 학습 목표
- **선형회귀, 로지스틱회귀, KNN, 의사결정트리, 랜덤포레스트** 이해와 실습
- Scikit-learn의 **표준 워크플로우** (fit/predict/score) 체화
- **Pipeline**을 이용한 전처리+모델 통합
- 각 모델의 **강점/약점**과 **하이퍼파라미터** 감 잡기


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

---

## 1. Scikit-learn 표준 워크플로우

```
  데이터 준비
      ↓
  train_test_split
      ↓
  모델 선택 & 인스턴스 생성
      ↓
  model.fit(X_train, y_train)     ← 학습
      ↓
  model.predict(X_test)           ← 예측
      ↓
  평가 (accuracy, rmse, ...)
```

이 패턴은 **모든 scikit-learn 모델**에 동일합니다. 한 번만 익히면 평생 씁니다.

In [ ]:
# 가장 기본적인 ML 코드 (5줄)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(X_tr, y_tr)
acc = accuracy_score(y_te, model.predict(X_te))
print(f"Accuracy: {acc:.4f}")

---

## 2. 선형 회귀 (Linear Regression)

### 직관
`y = w₁x₁ + w₂x₂ + ... + b` 형태의 **직선/평면**을 데이터에 맞춥니다.

### 언제 쓰나
- 연속 타깃 예측 (집값, 매출)
- 해석 쉬운 베이스라인
- 피처와 타깃이 **선형 관계**일 때

### 단점
- 비선형 관계 못 잡음
- 이상치에 민감

In [ ]:
from sklearn.linear_model import LinearRegression

# 합성 데이터
X, y, true_coef = make_regression(n_samples=200, n_features=3, noise=10, coef=True, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

lr = LinearRegression()
lr.fit(X_tr, y_tr)

print(f"coef  : {lr.coef_}")
print(f"true  : {true_coef}")
print(f"inter : {lr.intercept_:.3f}")
print(f"R²    : {lr.score(X_te, y_te):.4f}")

### Ridge / Lasso - 과적합 방지용 정규화

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Ridge: L2 penalty (계수를 작게)
# Lasso: L1 penalty (작은 계수는 0으로 → feature selection 효과)

for name, Model in [('Ridge', Ridge), ('Lasso', Lasso)]:
    m = Model(alpha=1.0)
    m.fit(X_tr, y_tr)
    print(f"{name:5s} → R²={m.score(X_te, y_te):.4f}, coef={m.coef_.round(2)}")

> 💡 **Kaggle 팁**: 선형모델은 빠른 베이스라인과 **스태킹의 meta-learner**로 자주 쓰입니다.

---

## 3. 로지스틱 회귀 (Logistic Regression)

### 직관
선형회귀 결과를 **시그모이드**로 눌러서 0~1 확률로 만든 것.

$$P(y=1|x) = \sigma(wx + b) = \frac{1}{1 + e^{-(wx+b)}}$$

### 언제 쓰나
- 이진 분류 베이스라인
- 확률 출력이 필요할 때
- 해석이 중요할 때 (의료, 금융)

In [ ]:
X, y = make_classification(n_samples=1000, n_features=5, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)[:, 1]  # 1 클래스 확률

print(f"Accuracy: {accuracy_score(y_te, y_pred):.4f}")
print(f"AUC     : {roc_auc_score(y_te, y_prob):.4f}")
print(f"Coef    : {clf.coef_[0].round(3)}")

### 시그모이드 직접 그려보기

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
plt.figure(figsize=(8, 3))
plt.plot(z, sigmoid(z), lw=2)
plt.axhline(0.5, ls='--', c='red', alpha=0.5)
plt.title('Sigmoid'); plt.xlabel('z'); plt.ylabel('σ(z)'); plt.grid(True); plt.show()

---

## 4. K-Nearest Neighbors (KNN)

### 직관
**"비슷한 이웃의 라벨을 따라가라"** — 새 데이터와 가장 가까운 K개 학습 샘플의 다수결로 예측.

### 언제 쓰나
- 데이터가 작고 저차원일 때
- 베이스라인이나 스태킹 재료

### 단점
- 샘플/차원 많으면 매우 느림 (예측시 전체 거리 계산)
- 스케일에 매우 민감 → **반드시 StandardScaler 필요**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 스케일링 전 vs 후
for scale in [False, True]:
    X_tr_, X_te_ = X_tr.copy(), X_te.copy()
    if scale:
        sc = StandardScaler(); X_tr_ = sc.fit_transform(X_tr_); X_te_ = sc.transform(X_te_)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_tr_, y_tr)
    print(f"Scaled={scale} → Acc={knn.score(X_te_, y_te):.4f}")

In [ ]:
# K값에 따른 성능 변화
ks = [1, 3, 5, 10, 20, 50, 100]
accs_tr, accs_te = [], []

sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)
for k in ks:
    knn = KNeighborsClassifier(k).fit(X_tr_s, y_tr)
    accs_tr.append(knn.score(X_tr_s, y_tr))
    accs_te.append(knn.score(X_te_s, y_te))

plt.figure(figsize=(8, 4))
plt.plot(ks, accs_tr, 'o-', label='train')
plt.plot(ks, accs_te, 'o-', label='test')
plt.xlabel('K'); plt.ylabel('accuracy'); plt.xscale('log')
plt.title('KNN: K vs accuracy'); plt.legend(); plt.grid(True); plt.show()

> 💡 K가 작을수록 **overfitting**, 크면 **underfitting**. 보통 √N 근처가 시작점.

---

## 5. 의사결정트리 (Decision Tree)

### 직관
**질문을 반복**해서 데이터를 나눔. `if age < 30 then ... else ...` 구조.

### 장점
- 해석 매우 쉬움 (트리 시각화 가능)
- 스케일링 불필요
- 범주형·수치형 혼합 OK

### 단점
- **단일 트리는 쉽게 과적합** → 앙상블(RF, XGBoost)로 해결

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=3, random_state=0)
dt.fit(X_tr, y_tr)

print(f"Train: {dt.score(X_tr, y_tr):.4f}")
print(f"Test : {dt.score(X_te, y_te):.4f}")

plt.figure(figsize=(15, 6))
plot_tree(dt, filled=True, rounded=True, feature_names=[f'f{i}' for i in range(5)])
plt.show()

In [ ]:
# 깊이에 따른 과적합 관찰
depths = list(range(1, 21))
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    tr.append(m.score(X_tr, y_tr))
    te.append(m.score(X_te, y_te))

plt.figure(figsize=(9, 4))
plt.plot(depths, tr, 'o-', label='train')
plt.plot(depths, te, 'o-', label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('Decision Tree: depth vs acc (overfitting 관찰)'); plt.legend(); plt.grid(True); plt.show()

---

## 6. 랜덤 포레스트 (Random Forest)

### 직관
**트리 100개를 각기 다른 부트스트랩 샘플·피처 부분집합으로 학습**하고 다수결.

### 왜 강력한가
- 개별 트리의 과적합을 평균내서 **분산 감소**
- 피처 중요도 자동 계산
- 하이퍼파라미터 튜닝 없이도 성능 좋음 → **첫 모델로 최적**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=0, n_jobs=-1)
rf.fit(X_tr, y_tr)
print(f"Train: {rf.score(X_tr, y_tr):.4f}")
print(f"Test : {rf.score(X_te, y_te):.4f}")
print(f"AUC  : {roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]):.4f}")

In [ ]:
# 피처 중요도
importance = pd.Series(rf.feature_importances_, index=[f'f{i}' for i in range(5)])
importance.sort_values().plot(kind='barh', figsize=(7, 3), color='steelblue')
plt.title('Feature Importance (Random Forest)'); plt.show()

### 주요 하이퍼파라미터

| 파라미터 | 설명 | 튜닝 팁 |
|----------|------|---------|
| `n_estimators` | 트리 개수 | 많을수록 좋음 (500~1000) |
| `max_depth` | 각 트리 최대 깊이 | None=제한없음, 10~30 자주 씀 |
| `max_features` | 분할 시 사용할 피처 수 | `sqrt`(분류) / `log2` 시작 |
| `min_samples_leaf` | 리프 최소 샘플 | 크게 하면 규제 효과 |

---

## 7. Pipeline — 전처리 + 모델 통합

### 왜 중요한가
- 전처리를 train/test에 **일관되게** 적용
- Cross-Validation 시 **데이터 누설 방지**
- 코드 재사용성 증가

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000))
])

pipe.fit(X_tr, y_tr)
print(f"Pipeline Accuracy: {pipe.score(X_te, y_te):.4f}")

### ColumnTransformer - 수치/범주 다르게 처리

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# 샘플 mixed 데이터
df = pd.DataFrame({
    'age':  [25, 32, np.nan, 45, 31],
    'fare': [7.25, 71.2, 8.05, 53.1, 8.05],
    'sex':  ['M', 'F', 'F', 'M', 'F'],
    'emb':  ['S', 'C', 'S', np.nan, 'Q'],
})
y_df = pd.Series([0, 1, 1, 0, 1])

num_cols = ['age', 'fare']
cat_cols = ['sex', 'emb']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('scale', StandardScaler())])

cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

final_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=1000))
])

final_pipe.fit(df, y_df)
print("Fitted! Predict:", final_pipe.predict(df))

---

## 8. 모델 비교 함수 (Kaggle 필수 템플릿)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           flip_y=0.05, random_state=0)

models = {
    'LogReg':   Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'KNN':      Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(5))]),
    'Tree':     DecisionTreeClassifier(max_depth=5, random_state=0),
    'RF':       RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    'GBM':      GradientBoostingClassifier(random_state=0),
}

results = []
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
    results.append({'model': name, 'mean_auc': scores.mean(), 'std': scores.std()})

pd.DataFrame(results).sort_values('mean_auc', ascending=False).round(4)

---

## 📝 Day 10-11 체크리스트
- [ ] 선형/로지스틱/KNN/Tree/RF 기본 사용 가능
- [ ] fit / predict / score 패턴 체화
- [ ] Pipeline + ColumnTransformer 사용
- [ ] cross_val_score로 모델 비교

다음 노트북에서 **Feature Engineering**을 본격적으로 배웁니다. 여기서 Kaggle 등수가 결정됩니다.